## HIS with knn and svm

"Hyperparameters were selected by exhaustive grid search using 3-fold stratified cross-validation on the outer training fold. The bandwidth ℓ was searched over 7 log-spaced values in [10⁻³, 1], γ over 6 log-spaced values in [10⁻⁴, 10⁻¹], C over {0.01, 0.1, 1, 10, 100}, and k over {1, 3, 5, 7, 9,11}. The distance matrix was computed once per ℓ value and reused across all classifier-specific hyperparameter combinations."

In [ ]:
# !pip install -q scikit-learn matplotlib seaborn pandas numpy scipy numba
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import scipy.linalg
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    precision_score, recall_score, f1_score,
)
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Numba JIT — accelerates the innermost kernel evaluation loop.
# Falls back gracefully if Numba is not installed.
# ---------------------------------------------------------------------------
try:
    from numba import njit

    @njit(cache=True)
    def _chol_solve(L, d):
        """Forward substitution: solve L @ y = d."""
        n = d.shape[0]
        y = np.empty(n)
        for i in range(n):
            s = d[i]
            for j in range(i):
                s -= L[i, j] * y[j]
            y[i] = s / L[i, i]
        return y

    @njit(cache=True)
    def _log_det_chol(L):
        s = 0.0
        for i in range(L.shape[0]):
            s += np.log(L[i, i])
        return 2.0 * s

    _NUMBA = True
    print("Numba available — kernel evaluations will use JIT compilation.")
except ImportError:
    _NUMBA = False
    print("Numba not found — falling back to NumPy (slower). "
          "Install with: pip install numba")


# =============================================================================
# Configuration
# =============================================================================

# ---- Paths ------------------------------------------------------------------
# Change NxGy to whichever config you select from the heatmap.
CSV_PATH    = ".../hmm_results/N5G5/hmm_results.csv"
OUTPUT_BASE = ".../hmm_results/rkhs_classifiers"

# ---- HMM model parameters (must match CSV) ----------------------------------
N_STATES     = 5
N_COMPONENTS = 5
N_FEATURES   = None   # None = auto-detected from CSV column names

# ---- Config tag (auto-derived from CSV path; override if needed) ------------
_match     = re.search(r'(N\d+G\d+)', CSV_PATH)
CONFIG_TAG = _match.group(1) if _match else f"N{N_STATES}G{N_COMPONENTS}"


# =============================================================================
# Checkpoint manager
#
# Saves progress after every completed outer fold and every PERM_CKPT_EVERY
# permutations, so a Kaggle session timeout does not lose completed work.
# On restart, automatically resumes from the next un-run fold/permutation.
# =============================================================================

PERM_CKPT_EVERY = 100   # save permutation test progress every N permutations


class CheckpointManager:
    def __init__(self, output_dir, classifier):
        self.path = os.path.join(output_dir, f"checkpoint_{classifier}.pkl")

    def load(self):
        """Returns the saved state dict, or None if no checkpoint exists."""
        if os.path.exists(self.path):
            try:
                with open(self.path, 'rb') as f:
                    state = pickle.load(f)
                print(f"  [checkpoint] Found existing checkpoint: {self.path}")
                return state
            except Exception as exc:
                print(f"  [checkpoint] Failed to load checkpoint, "
                      f"starting fresh: {exc}")
                return None
        return None

    def save(self, state):
        """Atomically save state to disk (write to temp file, then rename)."""
        tmp_path = self.path + ".tmp"
        with open(tmp_path, 'wb') as f:
            pickle.dump(state, f)
        os.replace(tmp_path, self.path)

    def clear(self):
        if os.path.exists(self.path):
            os.remove(self.path)

# ---- Nested CV structure ----------------------------------------------------
N_OUTER_SPLITS  = 5
N_OUTER_REPEATS = 5    # 5 x 5 = 25 outer evaluations
N_INNER_SPLITS  = 3    # inner CV folds for grid search
OUTER_CV_BASE_SEED = 0  # repeat r uses seed BASE + r

# ---- Hyperparameter grid ----------------------------------------------------
# ell  : bandwidth of the RKHS base Gaussian-of-Gaussians kernel
#        shared between KNN and SVM — same values searched for both.
# gamma: SVM RBF envelope bandwidth (converts MMD² distance to kernel value)
#        only relevant for SVM.
# C    : SVM regularisation parameter.
# k    : number of neighbours for KNN (odd values only).

ELL_GRID   = np.logspace(-3, 0, 7).tolist()   # 7 values: 0.001 .. 1.0
GAMMA_GRID = np.logspace(-4, -1, 6).tolist()  # 6 values: 1e-4 .. 0.1
C_GRID     = [0.01, 0.1, 1.0, 10.0, 100.0]
K_GRID     = [1, 3, 5, 7, 9]

# ---- Kernel regularisation (SVM PSD correction) ----------------------------
# Applied after eigenvalue flooring: K_reg = K_floored + KERNEL_EPS * I
# Ensures strict positive definiteness for SVM optimisation convergence.
KERNEL_EPS = 1e-10

# ---- Permutation test -------------------------------------------------------
N_PERMUTATIONS   = 1000
PERMUTATION_SEED = 0

# ---- Bootstrap CI -----------------------------------------------------------
N_BOOTSTRAP = 2000
CI_ALPHA    = 0.95

# ---- Column names in the pipeline CSV ---------------------------------------
COL_SUBJECT_TYPE = "subject_type"   # 'CONTROL' or 'ADHD'
COL_SUBJECT_ID   = "subject_id"     # 'v107', 'v41p', etc.
ADHD_LABEL       = "ADHD"


# =============================================================================
# Stationary distribution
# =============================================================================

def compute_stationary(A, max_iter=2000, rel_tol=1e-8, abs_tol=1e-10, eps=1e-12):
    """Power-iteration stationary distribution of a row-stochastic matrix."""
    A  = np.asarray(A, dtype=float)
    rs = A.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    A  = A / rs
    K  = A.shape[0]
    pi = np.ones(K, dtype=float) / K
    for _ in range(max_iter):
        pi_new = pi @ A
        if np.linalg.norm(pi_new - pi, 1) <= rel_tol * np.linalg.norm(pi, 1) + abs_tol:
            pi = pi_new
            break
        pi = pi_new
    pi = np.maximum(pi, 0.0)
    s  = pi.sum()
    return pi / (s if s > 0 else (1.0 + eps))


# =============================================================================
# RKHS base kernel: Gaussian-of-Gaussians
#
# Notation (matches paper):
#   ell (ℓ) — bandwidth of the base kernel between Gaussian components.
#
# k(N(mu1,S1), N(mu2,S2); ell) = N(mu1-mu2 ; 0, S1+S2+ell*I)
#
# Computed in log-space via Cholesky for numerical stability.
# =============================================================================

def _k_gg_numpy(mu1, S1, mu2, S2, ell, eps=1e-9):
    """NumPy fallback for Gaussian-of-Gaussians kernel."""
    D  = mu1.shape[0]
    M  = S1 + S2 + max(ell, eps) * np.eye(D)
    M += eps * np.eye(D)
    try:
        L = np.linalg.cholesky(M)
    except np.linalg.LinAlgError:
        M += 1e-6 * np.eye(D)
        L  = np.linalg.cholesky(M)
    log_det  = 2.0 * np.sum(np.log(np.diag(L)))
    y        = scipy.linalg.solve_triangular(L, mu1 - mu2, lower=True)
    log_norm = -0.5 * (D * np.log(2.0 * np.pi) + log_det)
    quad     = -0.5 * float(y @ y)
    return float(np.exp(log_norm + quad))


def k_gauss_of_gaussians(mu1, S1, mu2, S2, ell, eps=1e-9):
    """
    Gaussian-of-Gaussians kernel with Numba JIT when available.

    Parameters
    ----------
    mu1, mu2 : array, shape (D,)   — component means
    S1,  S2  : array, shape (D,D)  — component covariances
    ell      : float               — RKHS base bandwidth (ℓ)
    """
    if not _NUMBA:
        return _k_gg_numpy(mu1, S1, mu2, S2, ell, eps)

    D  = mu1.shape[0]
    M  = S1 + S2 + max(ell, eps) * np.eye(D) + eps * np.eye(D)
    try:
        L = np.linalg.cholesky(M)
    except np.linalg.LinAlgError:
        M += 1e-6 * np.eye(D)
        L  = np.linalg.cholesky(M)
    log_det  = _log_det_chol(L)
    y        = _chol_solve(L, mu1 - mu2)
    log_norm = -0.5 * (D * np.log(2.0 * np.pi) + log_det)
    quad     = -0.5 * float(np.dot(y, y))
    return float(np.exp(log_norm + quad))


# =============================================================================
# MMD² distance between two HMM-GMM models
#
# Each HMM-GMM is collapsed to a mixture of Gaussians:
#   w_{s,k} = pi_stationary[s] * alpha[s,k]   (stationary-flow weight)
#
# MMD²(P,Q;ell) = E_PP[k] + E_QQ[k] - 2*E_PQ[k]
#
# where each expectation is a weighted sum over mixture components.
# =============================================================================

def model_to_components(model):
    """
    Collapse HMM-GMM to (weights, means, covariances) mixture representation.
    Weights are stationary-flow probabilities: w_{s,k} = pi_s[s] * alpha[s,k].
    """
    K, M  = model['n_states'], model['n_components']
    pi_s  = model['pi_stationary']
    alpha = model['alpha']
    mu    = model['mu']
    sigma = model['sigma']

    w   = np.array([pi_s[s] * alpha[s, k]
                    for s in range(K) for k in range(M)], dtype=float)
    MU  = np.array([mu[s, k]
                    for s in range(K) for k in range(M)], dtype=float)
    SIG = np.array([sigma[s, k]
                    for s in range(K) for k in range(M)], dtype=float)
    w_sum = w.sum()
    w     = w / w_sum if w_sum > 1e-12 else np.ones(len(w)) / len(w)
    return w, MU, SIG


def _mmd2(cP, cQ, ell):
    """
    MMD² between two mixture representations at bandwidth ell.

        MMD²(P,Q;ell) = k(P,P) + k(Q,Q) - 2*k(P,Q)

    where k(·,·) is the sum of pairwise Gaussian-of-Gaussians kernel
    evaluations weighted by mixture weights.
    """
    wP, MUP, SIGP = cP
    wQ, MUQ, SIGQ = cQ

    def cross(wA, MUA, SIGA, wB, MUB, SIGB):
        total = 0.0
        for i in range(len(wA)):
            for j in range(len(wB)):
                total += wA[i] * wB[j] * k_gauss_of_gaussians(
                    MUA[i], SIGA[i], MUB[j], SIGB[j], ell)
        return total

    kPP = cross(wP, MUP, SIGP, wP, MUP, SIGP)
    kQQ = cross(wQ, MUQ, SIGQ, wQ, MUQ, SIGQ)
    kPQ = cross(wP, MUP, SIGP, wQ, MUQ, SIGQ)
    return max(float(kPP + kQQ - 2.0 * kPQ), 0.0)


def compute_distance_matrix(models, ell, verbose=True):
    """
    Compute the full n×n symmetric MMD² distance matrix for bandwidth ell.

    This matrix is shared between KNN (uses it directly as a precomputed
    distance) and SVM (converts it to a kernel via distance_to_kernel).

    Parameters
    ----------
    models : list of model dicts
    ell    : float — RKHS base bandwidth (ℓ)
    """
    n     = len(models)
    D     = np.zeros((n, n), dtype=float)
    comps = [model_to_components(m) for m in models]

    if verbose:
        print(f"    MMD² distance matrix: n={n}, ell={ell:.4g} ...")

    for i in range(n):
        for j in range(i + 1, n):
            d        = _mmd2(comps[i], comps[j], ell)
            D[i, j]  = d
            D[j, i]  = d

    if verbose:
        print(f"    Done. min={D.min():.4g}  max={D.max():.4g}  "
              f"mean={D.mean():.4g}")
    return D


# =============================================================================
# SVM kernel construction
#
# Notation (matches paper):
#   ell   (ℓ) — RKHS base bandwidth (set in compute_distance_matrix)
#   gamma (γ) — SVM RBF envelope bandwidth (separate from ell)
#
# K_SVM(i,j) = exp( -MMD²(i,j; ell) / (2 * gamma²) )
#
# Regularisation:
#   1. Eigenvalue flooring: negative eigenvalues clipped to 0
#      (corrects structural non-PSD from the distance-to-kernel conversion)
#   2. Diagonal shift: K += KERNEL_EPS * I
#      (ensures strict PD for SVM solver convergence)
# =============================================================================

def distance_to_kernel(D, gamma, kernel_eps=KERNEL_EPS):
    """
    Convert MMD² distance matrix to a PSD kernel matrix for SVM.

    Parameters
    ----------
    D          : array (n,n) — MMD² distance matrix from compute_distance_matrix
    gamma      : float       — SVM RBF envelope bandwidth (γ), independent of ell
    kernel_eps : float       — diagonal regularisation after eigenvalue flooring
    """
    K = np.exp(-D / (2.0 * max(gamma, 1e-12) ** 2))

    # ---- PSD correction: eigenvalue flooring --------------------------------
    eigvals, eigvecs = np.linalg.eigh(K)
    n_negative = (eigvals < 0).sum()
    if n_negative > 0:
        eigvals = np.maximum(eigvals, 0.0)
        K       = eigvecs @ np.diag(eigvals) @ eigvecs.T
        K       = (K + K.T) / 2.0   # enforce exact symmetry after reconstruction

    # ---- Diagonal regularisation --------------------------------------------
    K += kernel_eps * np.eye(K.shape[0])
    return K


def check_kernel_psd(K):
    """Report eigenvalue statistics of a kernel matrix."""
    eigvals     = np.linalg.eigvalsh(K)
    min_eig     = float(eigvals.min())
    max_eig     = float(eigvals.max())
    n_negative  = int((eigvals < 0).sum())
    condition   = max_eig / max(abs(min_eig), 1e-12)
    print(f"    Kernel PSD check: min_eig={min_eig:.3e}  max_eig={max_eig:.3e}  "
          f"n_negative={n_negative}  condition={condition:.3e}")
    return min_eig >= -1e-10


# =============================================================================
# CSV loading and parameter extraction
# =============================================================================

def infer_n_features(df):
    idxs = []
    for c in df.columns:
        if c.startswith("gmm_mean_0_0_f"):
            try:
                idxs.append(int(c.split('f')[-1]))
            except ValueError:
                pass
    return (max(idxs) + 1) if idxs else 1


def extract_models_from_df(df, n_states, n_components, n_features):
    has_stat_pi = all(
        f"stationary_pi_{i}" in df.columns for i in range(n_states)
    )
    models = []
    for idx, row in df.iterrows():
        try:
            A = np.array([[row[f"A_{i}{j}"] for j in range(n_states)]
                          for i in range(n_states)], dtype=float)
            rs = A.sum(axis=1, keepdims=True)
            rs[rs == 0] = 1.0
            A = A / rs

            if has_stat_pi:
                pi_s = np.array([row[f"stationary_pi_{i}"]
                                 for i in range(n_states)], dtype=float)
                if not (np.isfinite(pi_s).all()
                        and pi_s.sum() > 1e-12
                        and (pi_s >= -1e-12).all()):
                    pi_s = compute_stationary(A)
                else:
                    pi_s = np.maximum(pi_s, 0.0)
                    pi_s /= pi_s.sum()
            else:
                pi_s = compute_stationary(A)

            alpha = np.zeros((n_states, n_components), dtype=float)
            mu    = np.zeros((n_states, n_components, n_features), dtype=float)
            sigma = np.zeros((n_states, n_components,
                              n_features, n_features), dtype=float)

            for s in range(n_states):
                for k in range(n_components):
                    alpha[s, k] = float(row[f"gmm_weight_{s}_{k}"])
                    for f in range(n_features):
                        mu[s, k, f] = float(row[f"gmm_mean_{s}_{k}_f{f}"])
                    Si = np.zeros((n_features, n_features), dtype=float)
                    for f1 in range(n_features):
                        for f2 in range(f1, n_features):
                            v = float(row[f"gmm_cov_{s}_{k}_f{f1}f{f2}"])
                            Si[f1, f2] = v
                            if f1 != f2:
                                Si[f2, f1] = v
                    Si += 1e-6 * np.eye(n_features)
                    try:
                        np.linalg.cholesky(Si)
                    except np.linalg.LinAlgError:
                        Si = np.eye(n_features) * 0.1
                    sigma[s, k] = Si

            alpha = np.maximum(alpha, 0.0)
            sums  = alpha.sum(axis=1, keepdims=True)
            sums[sums == 0] = 1.0
            alpha = alpha / sums

            s_type      = str(row[COL_SUBJECT_TYPE]).strip().upper()
            class_label = 1 if s_type == ADHD_LABEL.upper() else 0

            models.append({
                'subject_id'       : str(row[COL_SUBJECT_ID]),
                'subject_type'     : s_type,
                'class_label'      : class_label,
                'pi_stationary'    : pi_s,
                'alpha'            : alpha,
                'mu'               : mu,
                'sigma'            : sigma,
                'transition_matrix': A,
                'n_states'         : n_states,
                'n_components'     : n_components,
                'n_features'       : n_features,
            })
        except Exception as exc:
            print(f"  [WARNING] Skipping row {idx}: {exc}")
    return models


def load_dataset(csv_path, n_states, n_components, n_features=None):
    print(f"Loading: {csv_path}")
    df = pd.read_csv(csv_path)
    if n_features is None:
        n_features = infer_n_features(df)
        print(f"  Auto-detected n_features = {n_features}")
    n_adhd    = (df[COL_SUBJECT_TYPE].str.upper() == ADHD_LABEL.upper()).sum()
    n_control = len(df) - n_adhd
    print(f"  Total={len(df)}  ADHD={n_adhd}  CONTROL={n_control}")
    models  = extract_models_from_df(df, n_states, n_components, n_features)
    labels  = np.array([m['class_label'] for m in models], dtype=int)
    sids    = np.array([m['subject_id']  for m in models])
    print(f"  Models extracted: {len(models)}")
    return models, labels, sids


# =============================================================================
# Metric computation
# =============================================================================

def compute_metrics(y_true, y_pred, y_prob):
    """All reviewer-required metrics. Positive class = 1 (ADHD)."""
    cm             = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'accuracy'         : float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'sensitivity'      : float(sensitivity),
        'specificity'      : float(specificity),
        'precision'        : float(precision_score(y_true, y_pred, zero_division=0)),
        'recall'           : float(recall_score(y_true, y_pred, zero_division=0)),
        'f1'               : float(f1_score(y_true, y_pred, zero_division=0)),
        'auc_roc'          : float(roc_auc_score(y_true, y_prob)),
        'mcc'              : float(matthews_corrcoef(y_true, y_pred)),
        'confusion_matrix' : cm,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'n_test'           : len(y_true),
        'n_adhd_test'      : int(y_true.sum()),
        'n_control_test'   : int((1 - y_true).sum()),
    }


def bootstrap_ci(values, n_bootstrap=N_BOOTSTRAP, alpha=CI_ALPHA, seed=0):
    rng    = np.random.default_rng(seed)
    values = np.asarray(values)
    boots  = np.array([
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    lo = np.percentile(boots, 100 * (1 - alpha) / 2)
    hi = np.percentile(boots, 100 * (1 + alpha) / 2)
    return float(values.mean()), float(lo), float(hi)


# =============================================================================
# Inner grid search  (nested inside one outer fold)
#
# For KNN: grid over (ell, k)
# For SVM: grid over (ell, gamma, C)
#
# Distance matrix is computed once per unique ell value —
# no redundant recomputation across (gamma, C, k) combinations.
# =============================================================================

def inner_grid_search_knn(train_models, train_labels,
                           ell_grid, k_grid,
                           n_inner_splits, inner_seed,
                           fold_id):
    """
    Grid search for KNN: exhaustive search over (ell, k).
    Distance matrix computed once per ell value and reused across all k.

    Returns best_ell, best_k, best_score, results_df.
    """
    skf     = StratifiedKFold(n_splits=n_inner_splits, shuffle=True,
                               random_state=inner_seed)
    records = []

    for ell in ell_grid:
        D_inner = compute_distance_matrix(train_models, ell, verbose=False)
        for k in k_grid:
            fold_scores = []
            for tr_idx, va_idx in skf.split(train_models, train_labels):
                tr_D = D_inner[np.ix_(tr_idx, tr_idx)]
                va_D = D_inner[np.ix_(va_idx, tr_idx)]
                knn  = KNeighborsClassifier(
                    n_neighbors=k, metric='precomputed', weights='distance'
                )
                knn.fit(tr_D, train_labels[tr_idx])
                pred = knn.predict(va_D)
                fold_scores.append(
                    balanced_accuracy_score(train_labels[va_idx], pred)
                )
            records.append({
                'ell': ell, 'k': k,
                'mean_bacc': float(np.mean(fold_scores)),
                'std_bacc' : float(np.std(fold_scores)),
            })

    results_df = pd.DataFrame(records).sort_values(
        'mean_bacc', ascending=False).reset_index(drop=True)
    best       = results_df.iloc[0]
    print(f"       [Inner-KNN {fold_id}] Best: "
          f"ell={best['ell']:.4g}  k={int(best['k'])}  "
          f"val_bacc={best['mean_bacc']:.4f}")
    return float(best['ell']), int(best['k']), float(best['mean_bacc']), results_df


def inner_grid_search_svm(train_models, train_labels,
                           ell_grid, gamma_grid, C_grid,
                           n_inner_splits, inner_seed,
                           fold_id):
    """
    Grid search for SVM: exhaustive search over (ell, gamma, C).
    Distance matrix computed once per ell; kernel matrix once per (ell, gamma).

    Returns best_ell, best_gamma, best_C, best_score, results_df.
    """
    skf     = StratifiedKFold(n_splits=n_inner_splits, shuffle=True,
                               random_state=inner_seed)
    records = []

    for ell in ell_grid:
        D_inner = compute_distance_matrix(train_models, ell, verbose=False)
        for gamma in gamma_grid:
            K_inner = distance_to_kernel(D_inner, gamma)
            for C in C_grid:
                fold_scores = []
                for tr_idx, va_idx in skf.split(train_models, train_labels):
                    tr_K = K_inner[np.ix_(tr_idx, tr_idx)]
                    va_K = K_inner[np.ix_(va_idx, tr_idx)]
                    svm  = SVC(kernel='precomputed', C=C,
                               probability=True, random_state=inner_seed)
                    svm.fit(tr_K, train_labels[tr_idx])
                    pred = svm.predict(va_K)
                    fold_scores.append(
                        balanced_accuracy_score(train_labels[va_idx], pred)
                    )
                records.append({
                    'ell': ell, 'gamma': gamma, 'C': C,
                    'mean_bacc': float(np.mean(fold_scores)),
                    'std_bacc' : float(np.std(fold_scores)),
                })

    results_df = pd.DataFrame(records).sort_values(
        'mean_bacc', ascending=False).reset_index(drop=True)
    best       = results_df.iloc[0]
    print(f"       [Inner-SVM {fold_id}] Best: "
          f"ell={best['ell']:.4g}  gamma={best['gamma']:.4g}  "
          f"C={best['C']:.4g}  val_bacc={best['mean_bacc']:.4f}")
    return (float(best['ell']), float(best['gamma']), float(best['C']),
            float(best['mean_bacc']), results_df)


# =============================================================================
# Nested cross-validation — KNN
# =============================================================================

def nested_cv_knn(models, labels, subject_ids,
                  ell_grid, k_grid,
                  n_outer_splits, n_outer_repeats, n_inner_splits,
                  outer_cv_base_seed, checkpoint_mgr=None):
    n_total      = n_outer_splits * n_outer_repeats
    fold_records = []
    subj_tracker = defaultdict(
        lambda: {'n_test': 0, 'n_correct': 0, 'predictions': [], 'true': -1}
    )
    inner_results_all = []
    global_fold       = 0
    completed_folds   = set()   # fold_ids already done, from checkpoint

    # ---- Resume from checkpoint if available --------------------------------
    if checkpoint_mgr is not None:
        state = checkpoint_mgr.load()
        if state is not None and state.get('phase') == 'nested_cv':
            fold_records      = state['fold_records']
            subj_tracker      = defaultdict(
                lambda: {'n_test': 0, 'n_correct': 0,
                         'predictions': [], 'true': -1},
                state['subj_tracker'],
            )
            inner_results_all = state.get('inner_results_all', [])
            completed_folds   = {r['fold_id'] for r in fold_records}
            print(f"  [checkpoint] Resuming: {len(completed_folds)}/{n_total} "
                  f"folds already completed")

    print("\n" + "=" * 70)
    print(f"  NESTED CV — KNN  [{CONFIG_TAG}]")
    print(f"  {n_outer_splits} folds × {n_outer_repeats} repeats = {n_total} evaluations")
    print(f"  Inner grid: ell({len(ell_grid)}) × k({len(k_grid)}) "
          f"= {len(ell_grid)*len(k_grid)} combinations per fold")
    print(f"  Distance matrices per fold: {len(ell_grid)} (one per ell value)")
    print("=" * 70)

    for repeat in range(n_outer_repeats):
        outer_seed = outer_cv_base_seed + repeat
        skf        = StratifiedKFold(n_splits=n_outer_splits, shuffle=True,
                                      random_state=outer_seed)
        print(f"\n  -- Repeat {repeat+1}/{n_outer_repeats}  (seed={outer_seed}) --")

        for fold, (train_idx, test_idx) in enumerate(skf.split(models, labels)):
            global_fold += 1
            fold_id      = f"r{repeat+1}f{fold+1}"

            if fold_id in completed_folds:
                print(f"\n     Fold {fold+1}/{n_outer_splits}  "
                      f"[global {global_fold}/{n_total}]  id={fold_id}  "
                      f"[SKIP - already completed]")
                continue

            tr_models    = [models[i] for i in train_idx]
            te_models    = [models[i] for i in test_idx]
            tr_labels    = labels[train_idx]
            te_labels    = labels[test_idx]

            print(f"\n     Fold {fold+1}/{n_outer_splits}  "
                  f"[global {global_fold}/{n_total}]  id={fold_id}")
            print(f"       Train: {len(train_idx)}  "
                  f"(Control={(tr_labels==0).sum()}  ADHD={(tr_labels==1).sum()})")
            print(f"       Test : {len(test_idx)}  "
                  f"(Control={(te_labels==0).sum()}  ADHD={(te_labels==1).sum()})")

            # Inner grid search — outer test fold never seen here
            best_ell, best_k, inner_score, inner_df = inner_grid_search_knn(
                tr_models, tr_labels, ell_grid, k_grid,
                n_inner_splits, outer_seed, fold_id,
            )
            inner_results_all.append({'fold_id': fold_id, 'results': inner_df})

            # Outer evaluation
            all_m  = tr_models + te_models
            full_D = compute_distance_matrix(all_m, best_ell, verbose=False)
            n_tr   = len(tr_models)
            tr_D   = full_D[:n_tr, :n_tr]
            te_D   = full_D[n_tr:, :n_tr]

            knn = KNeighborsClassifier(
                n_neighbors=best_k, metric='precomputed', weights='distance'
            )
            knn.fit(tr_D, tr_labels)
            te_pred = knn.predict(te_D)
            te_prob = knn.predict_proba(te_D)[:, 1]

            metrics = compute_metrics(te_labels, te_pred, te_prob)
            metrics.update({
                'config_tag'    : CONFIG_TAG,
                'classifier'    : 'KNN',
                'fold_id'       : fold_id,
                'repeat'        : repeat,
                'fold'          : fold,
                'outer_seed'    : outer_seed,
                'best_ell'      : best_ell,
                'best_k'        : best_k,
                'inner_best_val': inner_score,
                'n_train'       : len(train_idx),
                'n_train_ctrl'  : int((tr_labels == 0).sum()),
                'n_train_adhd'  : int((tr_labels == 1).sum()),
            })
            fold_records.append(metrics)

            print(f"       [KNN] Acc={metrics['accuracy']:.4f}  "
                  f"BalAcc={metrics['balanced_accuracy']:.4f}  "
                  f"AUC={metrics['auc_roc']:.4f}  "
                  f"Sens={metrics['sensitivity']:.4f}  "
                  f"Spec={metrics['specificity']:.4f}  "
                  f"F1={metrics['f1']:.4f}  "
                  f"MCC={metrics['mcc']:.4f}")

            for i, gi in enumerate(test_idx):
                sid = subject_ids[gi]
                subj_tracker[sid]['n_test']    += 1
                subj_tracker[sid]['n_correct'] += int(te_pred[i] == te_labels[i])
                subj_tracker[sid]['predictions'].append(int(te_pred[i]))
                subj_tracker[sid]['true']       = int(te_labels[i])

            # ---- Checkpoint after every completed fold --------------------------
            if checkpoint_mgr is not None:
                checkpoint_mgr.save({
                    'phase'             : 'nested_cv',
                    'fold_records'      : fold_records,
                    'subj_tracker'      : dict(subj_tracker),
                    'inner_results_all' : inner_results_all,
                })
                print(f"       [checkpoint] Saved progress "
                      f"({len(fold_records)}/{n_total} folds)")

    return fold_records, dict(subj_tracker), inner_results_all


# =============================================================================
# Nested cross-validation — SVM
# =============================================================================

def nested_cv_svm(models, labels, subject_ids,
                  ell_grid, gamma_grid, C_grid,
                  n_outer_splits, n_outer_repeats, n_inner_splits,
                  outer_cv_base_seed, checkpoint_mgr=None):
    n_total      = n_outer_splits * n_outer_repeats
    fold_records = []
    subj_tracker = defaultdict(
        lambda: {'n_test': 0, 'n_correct': 0, 'predictions': [], 'true': -1}
    )
    inner_results_all = []
    global_fold       = 0
    completed_folds   = set()

    # ---- Resume from checkpoint if available --------------------------------
    if checkpoint_mgr is not None:
        state = checkpoint_mgr.load()
        if state is not None and state.get('phase') == 'nested_cv':
            fold_records      = state['fold_records']
            subj_tracker      = defaultdict(
                lambda: {'n_test': 0, 'n_correct': 0,
                         'predictions': [], 'true': -1},
                state['subj_tracker'],
            )
            inner_results_all = state.get('inner_results_all', [])
            completed_folds   = {r['fold_id'] for r in fold_records}
            print(f"  [checkpoint] Resuming: {len(completed_folds)}/{n_total} "
                  f"folds already completed")

    print("\n" + "=" * 70)
    print(f"  NESTED CV — SVM  [{CONFIG_TAG}]")
    print(f"  {n_outer_splits} folds × {n_outer_repeats} repeats = {n_total} evaluations")
    print(f"  Inner grid: ell({len(ell_grid)}) × gamma({len(gamma_grid)}) × "
          f"C({len(C_grid)}) = {len(ell_grid)*len(gamma_grid)*len(C_grid)} "
          f"combinations per fold")
    print(f"  Distance matrices per fold: {len(ell_grid)} (one per ell value)")
    print(f"  Kernel matrices per fold  : {len(ell_grid)*len(gamma_grid)} "
          f"(one per (ell,gamma) pair)")
    print(f"  Notation: ell (ℓ) = RKHS base bandwidth; "
          f"gamma (γ) = SVM RBF envelope bandwidth")
    print("=" * 70)

    for repeat in range(n_outer_repeats):
        outer_seed = outer_cv_base_seed + repeat
        skf        = StratifiedKFold(n_splits=n_outer_splits, shuffle=True,
                                      random_state=outer_seed)
        print(f"\n  -- Repeat {repeat+1}/{n_outer_repeats}  (seed={outer_seed}) --")

        for fold, (train_idx, test_idx) in enumerate(skf.split(models, labels)):
            global_fold += 1
            fold_id      = f"r{repeat+1}f{fold+1}"

            if fold_id in completed_folds:
                print(f"\n     Fold {fold+1}/{n_outer_splits}  "
                      f"[global {global_fold}/{n_total}]  id={fold_id}  "
                      f"[SKIP - already completed]")
                continue

            tr_models    = [models[i] for i in train_idx]
            te_models    = [models[i] for i in test_idx]
            tr_labels    = labels[train_idx]
            te_labels    = labels[test_idx]

            print(f"\n     Fold {fold+1}/{n_outer_splits}  "
                  f"[global {global_fold}/{n_total}]  id={fold_id}")
            print(f"       Train: {len(train_idx)}  "
                  f"(Control={(tr_labels==0).sum()}  ADHD={(tr_labels==1).sum()})")
            print(f"       Test : {len(test_idx)}  "
                  f"(Control={(te_labels==0).sum()}  ADHD={(te_labels==1).sum()})")

            # Inner grid search
            best_ell, best_gamma, best_C, inner_score, inner_df = \
                inner_grid_search_svm(
                    tr_models, tr_labels, ell_grid, gamma_grid, C_grid,
                    n_inner_splits, outer_seed, fold_id,
                )
            inner_results_all.append({'fold_id': fold_id, 'results': inner_df})

            # Outer evaluation
            all_m  = tr_models + te_models
            full_D = compute_distance_matrix(all_m, best_ell, verbose=False)
            n_tr   = len(tr_models)
            tr_D   = full_D[:n_tr, :n_tr]
            te_D   = full_D[n_tr:, :n_tr]

            tr_K = distance_to_kernel(tr_D, best_gamma)
            # Test-to-train kernel: no PSD requirement needed for prediction
            te_K = np.exp(-te_D / (2.0 * max(best_gamma, 1e-12) ** 2))

            check_kernel_psd(tr_K)

            svm = SVC(kernel='precomputed', C=best_C,
                      probability=True, random_state=outer_seed)
            svm.fit(tr_K, tr_labels)
            te_pred = svm.predict(te_K)
            te_prob = svm.predict_proba(te_K)[:, 1]

            metrics = compute_metrics(te_labels, te_pred, te_prob)
            metrics.update({
                'config_tag'    : CONFIG_TAG,
                'classifier'    : 'SVM',
                'fold_id'       : fold_id,
                'repeat'        : repeat,
                'fold'          : fold,
                'outer_seed'    : outer_seed,
                'best_ell'      : best_ell,
                'best_gamma'    : best_gamma,
                'best_C'        : best_C,
                'inner_best_val': inner_score,
                'n_train'       : len(train_idx),
                'n_train_ctrl'  : int((tr_labels == 0).sum()),
                'n_train_adhd'  : int((tr_labels == 1).sum()),
            })
            fold_records.append(metrics)

            print(f"       [SVM] Acc={metrics['accuracy']:.4f}  "
                  f"BalAcc={metrics['balanced_accuracy']:.4f}  "
                  f"AUC={metrics['auc_roc']:.4f}  "
                  f"Sens={metrics['sensitivity']:.4f}  "
                  f"Spec={metrics['specificity']:.4f}  "
                  f"F1={metrics['f1']:.4f}  "
                  f"MCC={metrics['mcc']:.4f}")

            for i, gi in enumerate(test_idx):
                sid = subject_ids[gi]
                subj_tracker[sid]['n_test']    += 1
                subj_tracker[sid]['n_correct'] += int(te_pred[i] == te_labels[i])
                subj_tracker[sid]['predictions'].append(int(te_pred[i]))
                subj_tracker[sid]['true']       = int(te_labels[i])

            # ---- Checkpoint after every completed fold --------------------------
            if checkpoint_mgr is not None:
                checkpoint_mgr.save({
                    'phase'             : 'nested_cv',
                    'fold_records'      : fold_records,
                    'subj_tracker'      : dict(subj_tracker),
                    'inner_results_all' : inner_results_all,
                })
                print(f"       [checkpoint] Saved progress "
                      f"({len(fold_records)}/{n_total} folds)")

    return fold_records, dict(subj_tracker), inner_results_all


# =============================================================================
# Permutation test  (shared for KNN and SVM)
# =============================================================================

def permutation_test(models, labels, classifier,
                     best_ell, best_k_or_params,
                     n_outer_splits, n_outer_repeats,
                     outer_cv_base_seed,
                     n_permutations, perm_seed,
                     observed_score, checkpoint_mgr=None):
    """
    Label-permutation significance test.

    Uses the median/mode hyperparameters from nested CV with the full
    distance matrix precomputed once and reused across all permutations.

    Parameters
    ----------
    classifier       : 'KNN' or 'SVM'
    best_ell         : float — RKHS base bandwidth (ℓ)
    best_k_or_params : int (KNN: k) or dict (SVM: {'gamma': g, 'C': c})
    observed_score   : float — mean balanced accuracy from nested CV
    checkpoint_mgr   : CheckpointManager — saves progress every
                       PERM_CKPT_EVERY permutations, including RNG state
                       for exact bitwise resumability.
    """
    rng           = np.random.default_rng(perm_seed)
    perm_scores   = np.zeros(n_permutations, dtype=float)
    start_p       = 0

    # ---- Resume from checkpoint if available --------------------------------
    if checkpoint_mgr is not None:
        state = checkpoint_mgr.load()
        if state is not None and state.get('phase') == 'permutation_test':
            perm_scores = state['perm_scores']
            start_p     = state['next_p']
            rng         = state['rng_state']   # restore exact RNG state
            print(f"  [checkpoint] Resuming permutation test from "
                  f"{start_p}/{n_permutations}")

    print(f"\n  Permutation test [{classifier}]: "
          f"n={n_permutations}  ell={best_ell:.4g}")
    if classifier == 'KNN':
        print(f"  Fixed params: k={best_k_or_params}")
    else:
        print(f"  Fixed params: gamma={best_k_or_params['gamma']:.4g}  "
              f"C={best_k_or_params['C']:.4g}")
    print(f"  Observed balanced accuracy: {observed_score:.4f}")
    print("  Pre-computing full distance matrix...")

    full_D = compute_distance_matrix(models, best_ell, verbose=True)
    if classifier == 'SVM':
        full_K = distance_to_kernel(full_D, best_k_or_params['gamma'])

    for p in range(start_p, n_permutations):
        perm_labels = rng.permutation(labels)
        fold_baccs  = []

        for repeat in range(n_outer_repeats):
            skf = StratifiedKFold(
                n_splits=n_outer_splits, shuffle=True,
                random_state=outer_cv_base_seed + repeat,
            )
            for train_idx, test_idx in skf.split(models, perm_labels):
                if classifier == 'KNN':
                    tr_D = full_D[np.ix_(train_idx, train_idx)]
                    te_D = full_D[np.ix_(test_idx,  train_idx)]
                    clf  = KNeighborsClassifier(
                        n_neighbors=best_k_or_params,
                        metric='precomputed', weights='distance',
                    )
                    clf.fit(tr_D, perm_labels[train_idx])
                    pred = clf.predict(te_D)
                else:
                    tr_K = full_K[np.ix_(train_idx, train_idx)]
                    te_D_sub = full_D[np.ix_(test_idx, train_idx)]
                    te_K = np.exp(
                        -te_D_sub / (2.0 * max(best_k_or_params['gamma'], 1e-12) ** 2)
                    )
                    clf = SVC(kernel='precomputed',
                              C=best_k_or_params['C'],
                              probability=False,
                              random_state=outer_cv_base_seed)
                    clf.fit(tr_K, perm_labels[train_idx])
                    pred = clf.predict(te_K)

                fold_baccs.append(
                    balanced_accuracy_score(perm_labels[test_idx], pred)
                )

        perm_scores[p] = float(np.mean(fold_baccs))

        # ---- Checkpoint every PERM_CKPT_EVERY permutations -------------------
        if checkpoint_mgr is not None and (p + 1) % PERM_CKPT_EVERY == 0:
            checkpoint_mgr.save({
                'phase'      : 'permutation_test',
                'perm_scores': perm_scores,
                'next_p'     : p + 1,
                'rng_state'  : rng,
            })
            print(f"    [checkpoint] Saved progress ({p+1}/{n_permutations})")

        if (p + 1) % 200 == 0:
            running_p = (perm_scores[:p+1] >= observed_score).mean()
            print(f"    {p+1}/{n_permutations}  running p={running_p:.4f}")

    p_value = float((perm_scores >= observed_score).mean())
    sig     = 'significant' if p_value < 0.05 else 'not significant'
    print(f"  p-value = {p_value:.4f}  ({sig} at alpha=0.05)")
    return perm_scores, p_value


# =============================================================================
# Aggregate results and bootstrap CIs
# =============================================================================

METRIC_KEYS = [
    'accuracy', 'balanced_accuracy', 'sensitivity', 'specificity',
    'precision', 'recall', 'f1', 'auc_roc', 'mcc',
]


def aggregate_results(fold_records):
    summary_rows = []
    for mk in METRIC_KEYS:
        vals = np.array([r[mk] for r in fold_records])
        mean, lo, hi = bootstrap_ci(vals)
        summary_rows.append({
            'config_tag': CONFIG_TAG,
            'classifier': fold_records[0].get('classifier', ''),
            'metric'    : mk,
            'mean'      : round(mean,           4),
            'std'       : round(float(vals.std()), 4),
            'ci_lower'  : round(lo,              4),
            'ci_upper'  : round(hi,              4),
            'min'       : round(float(vals.min()), 4),
            'max'       : round(float(vals.max()), 4),
            'n_folds'   : len(vals),
        })

    pooled_cm = sum(r['confusion_matrix'] for r in fold_records)

    fold_df = pd.DataFrame([{
        'config_tag'      : r['config_tag'],
        'classifier'      : r.get('classifier', ''),
        'fold_id'         : r['fold_id'],
        'repeat'          : r['repeat'],
        'fold'            : r['fold'],
        'outer_seed'      : r['outer_seed'],
        'best_ell'        : r['best_ell'],
        'best_k'          : r.get('best_k', np.nan),
        'best_gamma'      : r.get('best_gamma', np.nan),
        'best_C'          : r.get('best_C', np.nan),
        'inner_best_val'  : r['inner_best_val'],
        'n_test'          : r['n_test'],
        'n_test_control'  : r['n_control_test'],
        'n_test_adhd'     : r['n_adhd_test'],
        'n_train'         : r['n_train'],
        'n_train_control' : r['n_train_ctrl'],
        'n_train_adhd'    : r['n_train_adhd'],
        'tp': r['tp'], 'tn': r['tn'], 'fp': r['fp'], 'fn': r['fn'],
        **{mk: r[mk] for mk in METRIC_KEYS},
    } for r in fold_records])

    return {
        'summary'  : pd.DataFrame(summary_rows),
        'fold_df'  : fold_df,
        'pooled_cm': pooled_cm,
    }


def build_subject_df(subject_tracker, classifier):
    rows = []
    for sid, info in subject_tracker.items():
        preds = info['predictions']
        rows.append({
            'config_tag'          : CONFIG_TAG,
            'classifier'          : classifier,
            'subject_id'          : sid,
            'true_label'          : info['true'],
            'true_class'          : 'ADHD' if info['true'] == 1 else 'Control',
            'n_folds_tested'      : info['n_test'],
            'n_correct'           : info['n_correct'],
            'accuracy_rate'       : round(info['n_correct'] / info['n_test'], 4)
                                    if info['n_test'] > 0 else 0.0,
            'pred_adhd_rate'      : round(sum(preds) / len(preds), 4)
                                    if preds else 0.0,
            'consistently_correct': info['n_correct'] == info['n_test'],
            'never_correct'       : info['n_correct'] == 0,
        })
    return pd.DataFrame(rows).sort_values(
        ['true_class', 'accuracy_rate'], ascending=[True, False]
    ).reset_index(drop=True)


# =============================================================================
# Printing
# =============================================================================

def print_summary(summary_df, pooled_cm, p_value, observed_score,
                  classifier, n_outer_splits, n_outer_repeats):
    n_folds = n_outer_splits * n_outer_repeats
    print("\n" + "=" * 70)
    print(f"  NESTED CV RESULTS [{CONFIG_TAG}] [{classifier}]  "
          f"({n_outer_splits}x{n_outer_repeats} = {n_folds} folds)")
    print(f"  Bootstrap CI: {int(CI_ALPHA*100)}%  (n={N_BOOTSTRAP})")
    print("=" * 70)
    print(f"\n  {'Metric':<22}  {'Mean':>7}  {'Std':>7}  "
          f"{'CI Lower':>9}  {'CI Upper':>9}  {'Min':>7}  {'Max':>7}")
    print("  " + "-" * 68)
    for _, row in summary_df.iterrows():
        print(f"  {row['metric']:<22}  {row['mean']:>7.4f}  {row['std']:>7.4f}  "
              f"{row['ci_lower']:>9.4f}  {row['ci_upper']:>9.4f}  "
              f"{row['min']:>7.4f}  {row['max']:>7.4f}")
    tn = pooled_cm[0,0]; fp = pooled_cm[0,1]
    fn = pooled_cm[1,0]; tp = pooled_cm[1,1]
    total = pooled_cm.sum()
    print(f"\n  Pooled confusion matrix (sum over {n_folds} folds, "
          f"total={total} predictions):")
    print(f"                   Pred Control  Pred ADHD")
    print(f"  True Control     {tn:<12}  {fp:<10}")
    print(f"  True ADHD        {fn:<12}  {tp:<10}")
    print(f"\n  Permutation test (n={N_PERMUTATIONS}):")
    print(f"    Observed balanced accuracy : {observed_score:.4f}")
    p_str = '<0.001' if p_value < 0.001 else f'{p_value:.4f}'
    sig   = '*significant*' if p_value < 0.05 else 'not significant'
    print(f"    p-value                    : {p_str}  ({sig} at alpha=0.05)")


# =============================================================================
# Plots
# =============================================================================

def save_plots(agg, perm_scores, p_value, observed_score,
               subject_df, output_dir, classifier):
    os.makedirs(output_dir, exist_ok=True)
    summary_df = agg['summary']
    fold_df    = agg['fold_df']
    pooled_cm  = agg['pooled_cm']
    tag        = f"[{CONFIG_TAG}][{classifier}]"

    # Figure 1 — metric summary with CI
    fig, ax = plt.subplots(figsize=(12, 5))
    x     = np.arange(len(METRIC_KEYS))
    sm    = summary_df.set_index('metric')
    means = sm.loc[METRIC_KEYS, 'mean'].values
    los   = sm.loc[METRIC_KEYS, 'ci_lower'].values
    his   = sm.loc[METRIC_KEYS, 'ci_upper'].values
    ax.bar(x, means, color='#4C72B0', alpha=0.8, zorder=3)
    ax.errorbar(x, means, yerr=np.array([means-los, his-means]),
                fmt='none', color='black', capsize=5, linewidth=1.5, zorder=4)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', '\n') for m in METRIC_KEYS], fontsize=9)
    ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
    ax.set_title(f"{tag} Metric summary — {int(CI_ALPHA*100)}% bootstrap CI",
                 fontsize=11)
    ax.axhline(0.5, color='grey', linewidth=0.8, linestyle='--')
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/metric_summary.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 2 — per-fold balanced accuracy
    fig, ax = plt.subplots(figsize=(14, 5))
    for rep in fold_df['repeat'].unique():
        sub = fold_df[fold_df['repeat'] == rep].sort_values('fold')
        xr  = sub['fold'].values + rep * (N_OUTER_SPLITS + 1)
        ax.plot(xr, sub['balanced_accuracy'].values, 'o-',
                label=f"Repeat {rep+1}", alpha=0.8, linewidth=1.5)
    ax.axhline(fold_df['balanced_accuracy'].mean(), color='black',
               linewidth=1.0, linestyle='--',
               label=f"Mean={fold_df['balanced_accuracy'].mean():.4f}")
    ax.set_ylabel("Balanced Accuracy"); ax.set_xlabel("Fold (by repeat)")
    ax.set_title(f"{tag} Per-fold balanced accuracy", fontsize=11)
    ax.legend(fontsize=8, framealpha=0.4); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/per_fold_bacc.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 3 — pooled confusion matrix
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(pooled_cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Control', 'ADHD'],
                yticklabels=['Control', 'ADHD'], ax=ax)
    ax.set_title(f"{tag}\nPooled confusion matrix", fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    fig.savefig(f"{output_dir}/pooled_cm.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 4 — permutation test
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(perm_scores, bins=40, color='#DD8452', alpha=0.75,
            edgecolor='white', label='Permuted scores')
    p_str = '<0.001' if p_value < 0.001 else f'{p_value:.3f}'
    ax.axvline(observed_score, color='#4C72B0', linewidth=2.5, linestyle='--',
               label=f'Observed={observed_score:.4f}  p={p_str}')
    ax.set_xlabel('Mean balanced accuracy'); ax.set_ylabel('Count')
    ax.set_title(f"{tag} Permutation test  (n={N_PERMUTATIONS})", fontsize=11)
    ax.legend(fontsize=9, framealpha=0.5); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/permutation_test.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 5 — subject stability
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = ['#1D9E75' if c == 'Control' else '#D85A30'
              for c in subject_df['true_class']]
    ax.bar(range(len(subject_df)), subject_df['accuracy_rate'],
           color=colors, alpha=0.8, zorder=3)
    ax.axhline(0.5, color='grey', linewidth=0.8, linestyle='--')
    ax.set_xticks(range(len(subject_df)))
    ax.set_xticklabels(subject_df['subject_id'], rotation=90, fontsize=6)
    ax.set_ylabel("Correct classification rate")
    ax.set_title(f"{tag} Per-subject stability", fontsize=11)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#1D9E75', label='Control'),
                       Patch(color='#D85A30', label='ADHD')], fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/subject_stability.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Figure 6 — hyperparameter distribution
    fig, axes = plt.subplots(1, 3 if classifier == 'SVM' else 2,
                              figsize=(14, 4))
    axes[0].hist(fold_df['best_ell'], bins=len(ELL_GRID),
                 color='#4C72B0', alpha=0.8, edgecolor='white')
    axes[0].set_xlabel('ell (ℓ) — RKHS base bandwidth')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f"{tag} Selected ell per outer fold")
    axes[0].set_xscale('log'); axes[0].grid(True, alpha=0.3)
    if classifier == 'KNN':
        k_counts = fold_df['best_k'].value_counts().sort_index()
        axes[1].bar(k_counts.index.astype(str), k_counts.values,
                    color='#DD8452', alpha=0.8)
        axes[1].set_xlabel('k (neighbours)')
        axes[1].set_ylabel('Count')
        axes[1].set_title(f"{tag} Selected k per outer fold")
        axes[1].grid(True, axis='y', alpha=0.3)
    else:
        axes[1].hist(fold_df['best_gamma'], bins=len(GAMMA_GRID),
                     color='#DD8452', alpha=0.8, edgecolor='white')
        axes[1].set_xlabel('gamma (γ) — SVM RBF envelope bandwidth')
        axes[1].set_ylabel('Count')
        axes[1].set_title(f"{tag} Selected gamma per outer fold")
        axes[1].set_xscale('log'); axes[1].grid(True, alpha=0.3)
        C_counts = fold_df['best_C'].value_counts().sort_index()
        axes[2].bar(C_counts.index.astype(str), C_counts.values,
                    color='#8172B2', alpha=0.8)
        axes[2].set_xlabel('C — SVM regularisation')
        axes[2].set_ylabel('Count')
        axes[2].set_title(f"{tag} Selected C per outer fold")
        axes[2].grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{output_dir}/hyperparameter_dist.png",
                dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f"  Figures saved to: {output_dir}")


# =============================================================================
# Run one classifier end-to-end
# =============================================================================

def run_classifier(models, labels, subject_ids, classifier, force_rerun=False):
    """
    Full pipeline for one classifier ('KNN' or 'SVM'):
        1. Nested CV with inner grid search
        2. Aggregate metrics + bootstrap CI
        3. Permutation test
        4. Save outputs

    If a COMPLETE result already exists on disk (fold_results.csv with
    exactly n_outer_splits * n_outer_repeats rows, plus permutation_scores.csv
    with n_permutations rows), this function skips recomputation entirely
    and loads the existing results instead — unless force_rerun=True.
    """
    assert classifier in ('KNN', 'SVM')
    out_dir = os.path.join(OUTPUT_BASE, CONFIG_TAG, classifier)
    os.makedirs(out_dir, exist_ok=True)
    checkpoint_mgr = CheckpointManager(out_dir, classifier)

    # ---- Skip entirely if a complete result already exists -------------------
    n_expected_folds = N_OUTER_SPLITS * N_OUTER_REPEATS
    fold_csv = os.path.join(out_dir, "fold_results.csv")
    perm_csv = os.path.join(out_dir, "permutation_scores.csv")
    perm_txt = os.path.join(out_dir, "permutation_result.txt")

    if not force_rerun and all(os.path.exists(p) for p in
                               (fold_csv, perm_csv, perm_txt)):
        try:
            existing_fold_df = pd.read_csv(fold_csv)
            existing_perm_df = pd.read_csv(perm_csv)
            if (len(existing_fold_df) == n_expected_folds
                    and len(existing_perm_df) == N_PERMUTATIONS):
                print(f"\n  [SKIP] Complete result already found for "
                      f"[{CONFIG_TAG}] [{classifier}] at {out_dir}")
                print(f"         fold_results.csv: {len(existing_fold_df)}/"
                      f"{n_expected_folds} folds")
                print(f"         permutation_scores.csv: "
                      f"{len(existing_perm_df)}/{N_PERMUTATIONS} permutations")
                print(f"         Pass force_rerun=True to recompute anyway.")

                existing_summary_df = pd.read_csv(
                    os.path.join(out_dir, "metric_summary.csv"))
                observed_ba = float(
                    existing_fold_df['balanced_accuracy'].mean())
                with open(perm_txt) as f:
                    txt = f.read()
                p_match = re.search(r'p-value\s*:\s*([0-9.]+)', txt)
                p_value = float(p_match.group(1)) if p_match else float('nan')

                # Reconstruct pooled confusion matrix from fold-level counts
                pooled_cm = np.array([
                    [existing_fold_df['tn'].sum(), existing_fold_df['fp'].sum()],
                    [existing_fold_df['fn'].sum(), existing_fold_df['tp'].sum()],
                ])

                print(f"\n  Loaded existing result: "
                      f"balanced_accuracy={observed_ba:.4f}  p={p_value:.4f}")

                return {
                    'fold_records': None,   # not reconstructed; CSV is the record
                    'subject_df'  : pd.read_csv(
                        os.path.join(out_dir, "subject_stability.csv")),
                    'agg': {
                        'summary'  : existing_summary_df,
                        'fold_df'  : existing_fold_df,
                        'pooled_cm': pooled_cm,
                    },
                    'perm_scores': existing_perm_df['balanced_accuracy'].values,
                    'p_value'    : p_value,
                    'observed_ba': observed_ba,
                }
        except Exception as exc:
            print(f"  [WARNING] Could not validate existing result "
                  f"({exc}) — recomputing from scratch.")

    # ---- Nested CV ----------------------------------------------------------
    if classifier == 'KNN':
        fold_records, subj_tracker, inner_all = nested_cv_knn(
            models, labels, subject_ids,
            ell_grid=ELL_GRID, k_grid=K_GRID,
            n_outer_splits=N_OUTER_SPLITS,
            n_outer_repeats=N_OUTER_REPEATS,
            n_inner_splits=N_INNER_SPLITS,
            outer_cv_base_seed=OUTER_CV_BASE_SEED,
            checkpoint_mgr=checkpoint_mgr,
        )
    else:
        fold_records, subj_tracker, inner_all = nested_cv_svm(
            models, labels, subject_ids,
            ell_grid=ELL_GRID, gamma_grid=GAMMA_GRID, C_grid=C_GRID,
            n_outer_splits=N_OUTER_SPLITS,
            n_outer_repeats=N_OUTER_REPEATS,
            n_inner_splits=N_INNER_SPLITS,
            outer_cv_base_seed=OUTER_CV_BASE_SEED,
            checkpoint_mgr=checkpoint_mgr,
        )

    # ---- Aggregate ----------------------------------------------------------
    agg         = aggregate_results(fold_records)
    subject_df  = build_subject_df(subj_tracker, classifier)
    observed_ba = float(agg['fold_df']['balanced_accuracy'].mean())

    # ---- Select permutation test hyperparameters ----------------------------
    # ell  : median across outer folds (robust to outliers on log scale)
    # k    : mode (KNN only)
    # gamma: median (SVM only)
    # C    : mode  (SVM only)
    best_ell_perm = float(agg['fold_df']['best_ell'].median())

    if classifier == 'KNN':
        best_k_perm      = int(agg['fold_df']['best_k'].mode()[0])
        perm_params      = best_k_perm
        perm_params_desc = f"ell={best_ell_perm:.4g}  k={best_k_perm}"
    else:
        best_gamma_perm  = float(agg['fold_df']['best_gamma'].median())
        best_C_perm      = float(agg['fold_df']['best_C'].mode()[0])
        perm_params      = {'gamma': best_gamma_perm, 'C': best_C_perm}
        perm_params_desc = (f"ell={best_ell_perm:.4g}  "
                            f"gamma={best_gamma_perm:.4g}  C={best_C_perm:.4g}")

    print(f"\n  Permutation test hyperparams: {perm_params_desc}")

    # The nested CV checkpoint is now stale (different phase) — the
    # permutation test reuses the same checkpoint file for its own phase.
    perm_scores, p_value = permutation_test(
        models, labels, classifier,
        best_ell=best_ell_perm,
        best_k_or_params=perm_params,
        n_outer_splits=N_OUTER_SPLITS,
        n_outer_repeats=N_OUTER_REPEATS,
        outer_cv_base_seed=OUTER_CV_BASE_SEED,
        n_permutations=N_PERMUTATIONS,
        perm_seed=PERMUTATION_SEED,
        observed_score=observed_ba,
        checkpoint_mgr=checkpoint_mgr,
    )

    # ---- All phases complete — clear checkpoint ------------------------------
    checkpoint_mgr.clear()
    print(f"  [checkpoint] Run complete — checkpoint cleared")

    # ---- Print --------------------------------------------------------------
    print_summary(agg['summary'], agg['pooled_cm'],
                  p_value, observed_ba, classifier,
                  N_OUTER_SPLITS, N_OUTER_REPEATS)

    # ---- Save ---------------------------------------------------------------
    agg['summary'].to_csv(f"{out_dir}/metric_summary.csv",    index=False)
    agg['fold_df'].to_csv(f"{out_dir}/fold_results.csv",      index=False)
    subject_df.to_csv(    f"{out_dir}/subject_stability.csv", index=False)

    perm_df = pd.DataFrame({
        'config_tag'       : CONFIG_TAG,
        'classifier'       : classifier,
        'permutation_index': np.arange(N_PERMUTATIONS),
        'balanced_accuracy': perm_scores,
    })
    perm_df.to_csv(f"{out_dir}/permutation_scores.csv", index=False)

    with open(f"{out_dir}/permutation_result.txt", 'w') as f:
        f.write(f"Config tag                 : {CONFIG_TAG}\n")
        f.write(f"Classifier                 : {classifier}\n")
        f.write(f"Observed balanced accuracy : {observed_ba:.6f}\n")
        f.write(f"n_permutations             : {N_PERMUTATIONS}\n")
        f.write(f"p-value                    : {p_value:.6f}\n")
        f.write(f"Significant at 0.05        : {p_value < 0.05}\n")
        f.write(f"Perm test params           : {perm_params_desc}\n")
        if classifier == 'SVM':
            f.write(f"Note: ell (base RKHS bandwidth) and gamma (SVM RBF envelope) "
                    f"are distinct parameters\n")

    save_plots(agg, perm_scores, p_value, observed_ba,
               subject_df, out_dir, classifier)

    print(f"\n  Outputs saved to: {out_dir}")
    return {
        'fold_records' : fold_records,
        'subject_df'   : subject_df,
        'agg'          : agg,
        'perm_scores'  : perm_scores,
        'p_value'      : p_value,
        'observed_ba'  : observed_ba,
    }


# =============================================================================
# Entry point
# =============================================================================

if __name__ == "__main__":

    print("=" * 70)
    print(f"  RKHS CLASSIFIERS — KNN and SVM  [{CONFIG_TAG}]")
    print(f"  Kernel: Gaussian-of-Gaussians MMD²")
    print(f"  Notation: ell (ℓ) = RKHS base bandwidth (shared)")
    print(f"            gamma (γ) = SVM RBF envelope (SVM only)")
    print(f"  ell grid  : {[f'{v:.3g}' for v in ELL_GRID]}")
    print(f"  gamma grid: {[f'{v:.3g}' for v in GAMMA_GRID]}")
    print(f"  C grid    : {C_GRID}")
    print(f"  k grid    : {K_GRID}")
    print("=" * 70)

    # Load dataset once — shared between both classifiers
    models, labels, subject_ids = load_dataset(
        csv_path=CSV_PATH, n_states=N_STATES,
        n_components=N_COMPONENTS, n_features=N_FEATURES,
    )

    # Run KNN
    results_knn = run_classifier(models, labels, subject_ids, 'KNN')

    # Run SVM
    results_svm = run_classifier(models, labels, subject_ids, 'SVM')

    # ---- Combined summary across both classifiers ---------------------------
    combined_df = pd.concat([
        results_knn['agg']['summary'],
        results_svm['agg']['summary'],
    ], ignore_index=True)
    combined_path = os.path.join(OUTPUT_BASE, CONFIG_TAG, 'combined_summary.csv')
    combined_df.to_csv(combined_path, index=False)

    print("\n" + "=" * 70)
    print(f"  COMPLETE  [{CONFIG_TAG}]")
    print("=" * 70)
    print(f"  KNN  balanced accuracy: {results_knn['observed_ba']:.4f}  "
          f"p={results_knn['p_value']:.4f}")
    print(f"  SVM  balanced accuracy: {results_svm['observed_ba']:.4f}  "
          f"p={results_svm['p_value']:.4f}")
    print(f"  Combined summary -> {combined_path}")
    print(f"  Output structure:")
    print(f"    {OUTPUT_BASE}/{CONFIG_TAG}/")
    print(f"      combined_summary.csv")
    print(f"      KNN/  metric_summary.csv  fold_results.csv  "
          f"subject_stability.csv  permutation_scores.csv  *.png")
    print(f"      SVM/  metric_summary.csv  fold_results.csv  "
          f"subject_stability.csv  permutation_scores.csv  *.png")